In [ ]:
import os
from dotenv import load_dotenv
load_dotenv()

os.environ["LANGSMITH_TRACING"] = "true"


In [ ]:
### Create the data points

from langsmith import Client

client = Client(api_key=os.getenv("LANGSMITH_API_KEY"))
print(client)


In [ ]:
dataset_name = "Simple Chatbot evaluation"
dataset = client.create_dataset(dataset_name = dataset_name)

client.create_examples(
    dataset_id=dataset.id,
    examples = [
    {
        "inputs": {"question": "What is LangChain used for?"},
        "outputs": {"answer": "LangChain is used to build applications powered by large language models, especially apps that connect LLMs with tools, data sources, memory, and workflows."},
    },
    {
        "inputs": {"question": "What is LangSmith used for?"},
        "outputs": {"answer": "LangSmith is used to trace, debug, evaluate, and monitor LLM applications."},
    },
    {
        "inputs": {"question": "What is a prompt template in LangChain?"},
        "outputs": {"answer": "A prompt template is a reusable structure for formatting inputs into prompts that can be sent to a language model."},
    },
    {
        "inputs": {"question": "What is retrieval augmented generation?"},
        "outputs": {"answer": "Retrieval augmented generation, or RAG, is a technique where relevant external documents are retrieved and passed to an LLM to help it answer more accurately."},
    },
    {
        "inputs": {"question": "Why should we evaluate LLM applications?"},
        "outputs": {"answer": "LLM applications should be evaluated to measure correctness, relevance, reliability, and regressions across different inputs."},
    },
]
)

In [ ]:
### Define models 

from langchain_ollama import ChatOllama

base_model = ChatOllama(model="llama3.1", temperature=0)
judge_model = ChatOllama(model="llama3.1", temperature=0)



In [ ]:
### Application being tested 
from langchain_core.messages import SystemMessage, HumanMessage

default_instructions : str = """Respond to the user's question correctly and concisely. Prefer one short sentence"""
def my_app(question: str):
    response = base_model.invoke(
        [
            HumanMessage(question),
            SystemMessage(default_instructions)
        ]
    )
    return response.content.strip()

In [ ]:
### Target function

def ls_target(inputs: dict):
    response = my_app(inputs["question"])
    return {
        "response" : response
    }

In [ ]:
### LLM as a judge
eval_instructions = """You are an expert professor specialized in grading students' answers to questions. Respond with only one of the following - CORRECT / INCORRECT"""

def correctness(inputs: dict, outputs: dict, reference_outputs: dict):
    prompt = f"""
    Question: {inputs["question"]}
    Reference Answer : {reference_outputs["answer"]}
    Predicted Answer : {outputs["response"]}

    Is the predicted answer factually correct?
    """
    answer = judge_model.invoke([
            HumanMessage(prompt),
            SystemMessage(eval_instructions)
        ]
    )

    answer = answer.content.strip().upper()

    return answer == "CORRECT"

In [ ]:
### LLM Evaluation

experiment_resuls = client.evaluate(
    ls_target, 
    dataset_name,
    evaluators=[correctness],
    experiment_prefix="ollama-llama31-chatbot"
)